# Veritas — Model Drift Simulation
Demonstrates how Evidently AI detects data drift in AI model outputs over time.

In [ ]:
import pandas as pd
import numpy as np
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, ClassificationPreset
from evidently.metrics import *

np.random.seed(42)
print('Veritas Drift Simulation loaded.')

In [ ]:
# Simulate reference data (model trained on this)
# Loan risk model — before price adjustment
n_reference = 1000

reference_data = pd.DataFrame({
    'credit_score': np.random.normal(680, 50, n_reference),
    'income': np.random.normal(75000, 15000, n_reference),
    'loan_amount': np.random.normal(25000, 8000, n_reference),
    'pricing_tier': np.random.choice(['standard', 'premium'], n_reference, p=[0.7, 0.3]),
    'debt_to_income': np.random.normal(0.35, 0.1, n_reference),
    'target': np.random.choice([0, 1], n_reference, p=[0.85, 0.15]),  # 15% default rate
})

print(f'Reference data: {len(reference_data)} samples')
print(f'Default rate (reference): {reference_data["target"].mean():.2%}')

In [ ]:
# Simulate current data (after price adjustment — distribution shifted)
# This is what the model is now seeing, but was NOT trained on
n_current = 500

current_data = pd.DataFrame({
    'credit_score': np.random.normal(650, 70, n_current),       # Scores shifted down
    'income': np.random.normal(68000, 20000, n_current),         # Income distribution shifted
    'loan_amount': np.random.normal(32000, 10000, n_current),    # Larger loans being requested
    'pricing_tier': np.random.choice(['standard', 'premium'], n_current, p=[0.4, 0.6]),  # More premium
    'debt_to_income': np.random.normal(0.48, 0.12, n_current),   # Higher debt ratio
    'target': np.random.choice([0, 1], n_current, p=[0.72, 0.28]),  # Default rate jumped to 28%
})

print(f'Current data: {len(current_data)} samples')
print(f'Default rate (current): {current_data["target"].mean():.2%}')
print(f'\nDrift Alert: Default rate increased from 15% to {current_data["target"].mean():.2%}')

In [ ]:
# Run Evidently drift report
report = Report(metrics=[
    DataDriftPreset(),
])

report.run(
    reference_data=reference_data.drop('target', axis=1),
    current_data=current_data.drop('target', axis=1),
)

report.save_html('drift_report.html')
print('Drift report saved to drift_report.html')
report.show()

In [ ]:
# Simulate model accuracy degradation over time
import matplotlib.pyplot as plt

weeks = list(range(1, 13))
accuracy = [
    0.91, 0.91, 0.90, 0.89, 0.88,  # Stable initially
    0.85, 0.82, 0.79, 0.77,        # Price adjustment kicks in — drift begins
    0.75, 0.74, 0.74                # Settled at degraded performance
]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(weeks, accuracy, marker='o', color='#8B1FA9', linewidth=2.5, markersize=8)
ax.axhline(y=0.80, color='red', linestyle='--', linewidth=1.5, label='Minimum acceptable accuracy (80%)')
ax.axvline(x=5, color='orange', linestyle='--', linewidth=1.5, label='Price adjustment event')
ax.fill_between(weeks, accuracy, 0.80, where=[a < 0.80 for a in accuracy], alpha=0.2, color='red', label='Below threshold')

ax.set_xlabel('Week', fontsize=12)
ax.set_ylabel('Model Accuracy', fontsize=12)
ax.set_title('Veritas Drift Detection — loan-risk-model-v2 Accuracy Over Time', fontsize=13, fontweight='bold')
ax.set_ylim(0.65, 1.0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('drift_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to drift_chart.png')